# Possession pipeline — summary and run

This notebook is the executable overview. The detailed reasoning is kept in the numbered guides next to it.

## What we do

1. Load and audit the supplied ATD, Smart Tagging, and metadata tables.
2. Apply the published graded-frame mask; output `X` outside the two graded intervals.
3. Build pitch distances, margins, zones, pressure, movement, reachability, and reception features.
4. Identify dead-ball context, comfortable-control anchors, flight, contest, occlusion, and reception scenarios.
5. Resolve bounded trajectories using previous anchors and later controlled reception evidence.
6. Protect strong `D` and `X` evidence, write H/A/D/X labels, and evaluate Macro-F1 and Spell-F1.

This is an offline trajectory-classification pipeline. A scenario may look forward to the later receiver, but the current H/A label is preserved until controlled reception evidence confirms a switch. The rule hierarchy is documented in `3b_RULE_BOOK.md`; Viterbi, HMM, and HSMM are registered alternative decoders rather than methods blindly stacked together.

## Modular implementation

The notebook calls the canonical runner; the runner composes these reusable functions:

- `ClosestTeamRuleModel.build_features()` — load data, audit empty columns, calculate frame time and player/ball distances;
- `add_temporal_features()` — lagged, rolling, persistence, ball movement, and pressure signals;
- `add_reachability_features()` — closing speed, projected path, arrival time, and arrival margin;
- `add_anchor_features()` and `add_special_anchor_features()` — stable H/A and restart evidence;
- `apply_anchor_hierarchy()` — choose restart control, pressure control, stable control, or no anchor in configured order;
- `PipelineSpec` and `apply_pipeline_spec()` — choose anchors, scenarios, transitions, decoders, and protection rules per benchmark;
- `add_reception_signal_features()` — local minima, deceleration, persistence, and receiver agreement;
- `classify_scenarios()` and `build_scenario_episodes()` — scenario context and connected trajectories;
- `dead_ball_boundary_evidence()` — protect D using Smart Tagging plus physical activity evidence;
- `resolve_aligned_reception_switch()` — bounded bidirectional reception-based switch resolution;
- `run_evaluation.py` — display public sanity metrics without fitting on public labels.

Thresholds and windows are configuration-backed in `../config/possession_rules.yaml`. Decoder alternatives are registered in `models/pipelines.py`: `none`, `viterbi`, `selective_viterbi`, `hmm`, `hsmm`, `bayesian_hmm`, and `duration_viterbi`.

In [ ]:
# Setup: keep paths and the two executable stages in one place.
from pathlib import Path
import subprocess
import sys
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'possession').exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
PIPELINE_MODULE = 'possession.run_interpolated_flight_experiment'
EVALUATOR = ROOT / 'evaluation' / 'run_evaluation.py'
RESULTS = ROOT / 'evaluation' / 'results'
print('project root:', ROOT)
print('pipeline module:', PIPELINE_MODULE)
print('evaluator:', EVALUATOR)

## Run the pipeline and show the results

This is the only execution step needed for the presentation. It generates the modular candidate submissions and then runs the evaluator across them. The displayed scoreboard is the evidence for which model is currently preferred.

In [ ]:
# Run feature engineering, scenario detection, resolvers, and evaluation.
pipeline = subprocess.run(
    [sys.executable, '-m', PIPELINE_MODULE],
    cwd=ROOT, capture_output=True, text=True, check=True,
)
print(pipeline.stdout)
evaluation = subprocess.run(
    [sys.executable, str(EVALUATOR), '--gt', 'public'],
    cwd=ROOT, capture_output=True, text=True, check=True,
)
print(evaluation.stdout)
scoreboard = pd.read_csv(ROOT / 'evaluation' / 'scoreboard.csv')
columns = [c for c in ['rank', 'submission', 'macroF1', 'spellF1'] if c in scoreboard.columns]
display(scoreboard[columns].head(12) if columns else scoreboard.head(12))
preferred = RESULTS / 'baseline_aligned_reception_switch.csv'
print('preferred submission:', preferred)
print('exists:', preferred.exists())

## Inspect one internal ball gap

This diagnostic plot is separate from submission labelling. Solid blue segments are observed ball coordinates; the orange dotted bridge is the missing interval; the purple dashed path is nearest-player geometry; red and green dots are player positions at one reference frame.

Smart Tagging is shown as a context clue only. It can cover preparation, flight, and reception together, so we inspect it selectively rather than copying its team label to every frame. ATD coordinates are normalised in the data and converted here to the supplied pitch in metres: x=0..105.3 and y=0..68.


In [ ]:
# Diagnostic only: visualise observed, missing, and nearest-player geometry.
from possession.baseline_pipeline import ClosestTeamRuleModel, slot_team_map
from possession.calculations.features import build_engineered_features
from possession.calculations.plotting import plot_trajectory_window, select_representative_ball_gap
from games import asset_paths
from possession.temporal_rules import load_rules
import matplotlib.pyplot as plt

rules = load_rules()
model = ClosestTeamRuleModel()
plot_features = build_engineered_features(model=model, rules=rules)
gap = select_representative_ball_gap(plot_features)
if gap is None:
    print('No internal ball-coordinate gap was found.')
else:
    gap_start, gap_end = gap
    team_by_slot = slot_team_map(asset_paths(model.game, model.data_dir, source='tactical')['atd_meta'])
    fig, ax = plot_trajectory_window(plot_features, max(0, gap_start - 25), gap_end + 25, team_by_slot=team_by_slot, reference_frame=(gap_start + gap_end) // 2)
    plt.show()


## Read the reasoning around the run

- [01 — EDA](./01_eda.ipynb)
- [02 — Feature engineering](./02_feature_engineering.ipynb)
- [03 — Scenario engineering](./03_scenario_engineering.ipynb)
- [04 — Anchors](./04_anchors_rulebook.ipynb)
- [05 — Transition markers](./05_transition_markers.ipynb)
- [06 — Viterbi model](./06_viterbi_model.ipynb)
- [08 — Experimental results](./08_experimental_results.ipynb)